In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

In [2]:
%pip install -q rpy2 "setuptools<81"  # xarray_schema (via spatialdata) still imports pkg_resources
# conda R has no compiler: compiled R packages from conda-forge, funkyheatmap from CRAN below
!mamba install -y -q -p {sys.prefix} -c conda-forge r-svglite r-showtext r-dplyr r-tibble r-ggplot2 r-rcolorbrewer r-ggforce r-jsonlite

Note: you may need to restart the kernel to use updated packages.


In [3]:
%load_ext rpy2.ipython

In [4]:
%%R
if (!requireNamespace("funkyheatmap", quietly = TRUE))
  install.packages("funkyheatmap", repos = "https://cloud.r-project.org")

In [5]:
main_dir = Path.cwd().parents[0] / 'data'

In [6]:
import sys
sys.path.insert(0, '/dss/dsshome1/0C/ra98gaq/Git/branch-metrics/cellseg-benchmark')

## load + format metric csvs

In [7]:
cohorts = ["aging", "ABCAtlas", "VizgenMouseBrain"]
exclude = ["Negative_Control_Visium"]  # methods left out of the tables
output_folders = ["general_stats", "assigned_transcripts", "cell_type_metrics", "marker_gene_metrics", "ficture", "cell_density", "z_integrity"]

dfs_by = {c: {csv.stem: pd.read_csv(csv, index_col=0)
              for name in output_folders
              for csv in (main_dir / "metrics" / c / name).glob("*.csv") if "vascular" not in csv.name}
          for c in cohorts}
dfs_by["merged"] = {k: pd.concat([d[k] for d in dfs_by.values() if k in d], ignore_index=True)
                    for k in set().union(*dfs_by.values())}
{c: len(d) for c, d in dfs_by.items()}

{'aging': 11, 'ABCAtlas': 9, 'VizgenMouseBrain': 9, 'merged': 11}

Per method, mean over samples (rows with `sample == "all"` ignored):
- general_stats (`cell_type_revised == "all"`): volume, elongation, PolyT/DAPI intensity
- Z integrity: % low-VSI signal inside the cells (signal > 2, VSI < 0.7), mean over samples
- assigned_transcript_counts: % assigned (QC'd cells), transcripts per cell
- clustering score: silhouette; cell type distribution: % undefined
- MECR: per-sample summary row, then size-adjusted (see metrics_pairwise_correlation)
- marker F1: mean over cell types; FICTURE F1: per-sample macro F1
- cell density: cells inside the tissue / tissue area, summed over samples (scripts/metrics/compute_cell_density.py)

## plot heatmap

In [8]:
import cellseg_benchmark._constants as _constants
from cellseg_benchmark.metrics import clean_method_name

def norm_fixed(s, lo, hi, invert=False):
    n = ((s - lo) / (hi - lo)).clip(0, 1)
    return 1 - n if invert else n

def norm01(s, ref):
    lo, hi = s[ref].min(), s[ref].max()
    return ((s - lo) / (hi - lo)).clip(0, 1) if hi > lo else s * 0 + 0.5

def per_method(df, cols, key="method"):
    return df.query("sample != 'all'").groupby(key)[cols].mean()

def build_benchmark_table(dfs, exclude=()):
    gs = dfs["general_stats"].query("cell_type_revised == 'all'")
    cd = dfs["cell_density"].query("sample != 'all'").groupby("method")
    density = cd.n_cells.sum() / cd.tissue_mm2.sum()
    at = dfs["assigned_transcript_counts"].groupby(["sample", "method"])[["assigned_count_qced", "total_count"]].sum()
    at["pct_assigned_qced"] = at.assigned_count_qced / at.total_count
    at["tpc"] = at.assigned_count_qced / gs.set_index(["sample", "method"]).n_cells
    at = at.reset_index()

    # NMP left out: correlates ~-0.9 with MECR and depends mostly on transcripts per cell
    # nmp = dfs["negative_marker_purity_all"].groupby("method")[["negative_marker_purity"]].mean()

    mecr = per_method(dfs["MECR_score_all"].query("gene1 == 'all'"), "MECR")
    tpc = per_method(at, "tpc").reindex(mecr.index)
    real = ~mecr.index.str.startswith("Negative_Control") & tpc.notna()
    fit = np.polyfit(np.log10(tpc[real]), mecr[real], 1)
    mf1 = (per_method(dfs["marker_f1_score_cell_type_revised"], "f1_score", ["sample", "method", "cell_type"])
           .groupby(["sample", "method"]).mean().groupby("method").mean())
    z = dfs["z_integrity_all"].query("sample != 'all_samples' and signal_threshold == 2 and vsi_threshold == 0.7").groupby("method").low_vsi_pct.mean()
    ff1 = dfs["ficture_f1"].query("cell_type == 'f1_macro_all' and sample != 'all_samples'").groupby("method").f1.mean()

    raw = pd.concat([
        per_method(gs, ["volume_final", "elongation", "intensities_PolyT", "intensities_DAPI", "n_cells"]),
        per_method(at, ["pct_assigned_qced"]),
        per_method(dfs["clustering_score_adata_integrated_cell_type_revised"], ["silhouette_score"]),
        per_method(dfs["cell_type_distribution_adata_integrated_cell_type_revised"], ["Undefined"]),
        mecr, (mecr - np.polyval(fit, np.log10(tpc))).rename("MECR_adj"),
        mf1.rename("marker_f1"), ff1.rename("ficture_f1_score"), z.rename("low_vsi_pct"), density.rename("cells_per_mm2"),
        tpc.rename("tpc"),
    ], axis=1).drop(index=list(exclude), errors="ignore")
    is_3d = raw.index.map(lambda m: any(m.startswith(p) for p in _constants.methods_3D)).to_numpy(bool)
    raw.loc[raw.index.str.startswith("Ficture_segments"), "ficture_f1_score"] = np.nan

    bounds = {
        "silhouette_score":            (-1.0, 1.0, False),
        # "Undefined":                 (0.0, 1.0, True),
        "MECR_adj":                    (-0.02, 0.02, True),
        "low_vsi_pct":                 (0.0, 100.0, True),
        "marker_f1":                   (0.0, 1.0, False),
        "ficture_f1_score":            (0.0, 1.0, False),
    }
    cats = {"clustering": ["silhouette_score"], "marker": ["MECR_adj", "marker_f1"],
            "ficture": ["ficture_f1_score"], "z": ["low_vsi_pct"]}
    descriptive = ["volume_final", "elongation", "intensities_DAPI", "cells_per_mm2", "pct_assigned_qced"]  # "intensities_PolyT"

    df = raw.copy()
    ref = ~df.index.str.startswith("Negative_Control")
    df["na_z"] = is_3d & df.low_vsi_pct.isna()
    df["na_ficture"] = (is_3d | df.index.str.startswith("Ficture_segments")) & df.ficture_f1_score.isna()
    df["density_label"] = df.cells_per_mm2.map(lambda x: f"{x / 1e3:.1f}")
    n = pd.DataFrame({c: norm01(norm_fixed(df[c], *b), ref) for c, b in bounds.items()})
    df["total_score"] = pd.DataFrame({k: n[v].mean(axis=1) for k, v in cats.items()}).mean(axis=1)
    for c, b in bounds.items():
        df[c] = norm01(norm_fixed(df[c], *b), ref)
    for c in descriptive:
        df[c] = norm01(df[c], ref)
    df = df.sort_values("total_score", ascending=False).rename_axis("id").reset_index()
    df["id"] = df.id.map(clean_method_name)
    return df, raw

metric_labels = pd.DataFrame(list(_constants.metric_names.items()), columns=["id", "name"])

/home/ubuntu/miniforge3/envs/analysis/lib/python3.12/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
/home/ubuntu/miniforge3/envs/analysis/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [29]:
%%R
plot_benchmark_heatmap <- function(df_funky, out_dir, metric_labels, family = "Arial", font_file = "../data/misc/Arial.ttf") {
  library(funkyheatmap)
  library(dplyr)
  library(tibble)
  library(ggplot2)
  library(RColorBrewer)

  sysfonts::font_add(family, path.expand(font_file))
  showtext::showtext_auto()
  showtext::showtext_opts(dpi = 600)
  update_geom_defaults("text", list(family = family))
  for (s in c("sep1", "sep2", "sep3")) df_funky[[s]] <- ""

  col_spec <- tribble(
    ~id,                          ~group,       ~geom,       ~palette,      ~name,          ~opts,
    "id",                         "",           "text",      NA,            "",             list(width = 8.8, hjust = 0),
    "total_score",                "overall",    "bar",       "overall",     "",             list(width = 2.8, draw_outline = FALSE),
    "sep1",                       "",           "text",      NA,            "",             list(width = 0.3),
    "silhouette_score",           "expression", "funkyrect", "expression",  "Silhouette",   list(),
    # "Undefined",                "expression", "funkyrect", "expression",  "% undefined",  list(),
    # "negative_marker_purity",   "expression", "funkyrect", "expression",  "NMP",          list(),
    "MECR_adj",                   "expression", "funkyrect", "expression",  "MECR",         list(),
    "marker_f1",                  "expression", "funkyrect", "expression",  "Marker F1",    list(),
    "sep2",                       "",           "text",      NA,            "",             list(width = 0.3),
    "ficture_f1_score",           "spatial",    "funkyrect", "spatial",     "Ficture F1",   list(),
    "low_vsi_pct",                "spatial",    "funkyrect", "spatial",     "Z integrity",  list(),
    "sep3",                       "",           "text",      NA,            "",             list(width = 0.8),
    "pct_assigned_qced",          "general",    "funkyrect", "transcripts", "% Assigned",   list(),
    "volume_final",               "general",    "funkyrect", "general",     "Volume",       list(),
    "elongation",                 "general",    "funkyrect", "general",     "Elongation",   list(),
    # "intensities_PolyT",        "general",    "funkyrect", "general",     "PolyT",        list(),
    "intensities_DAPI",           "general",    "funkyrect", "general",     "DAPI",         list(),
    "cells_per_mm2",              "general",    "rect",      "general_lt",  "Cells/mm²",    list(width = 1.1),
    "density_label",              "general",    "text",      NA,            "",             list(overlay = TRUE, size = 2.4, hjust = 0.5),
  )

  column_info <- col_spec %>%
    left_join(metric_labels, by = "id", suffix = c("", "_label")) %>%
    mutate(name = coalesce(name_label, name), name = ifelse(id == "cells_per_mm2", "Cells/mm² (×10³)", name),
           options = opts, legend = FALSE) %>%
    select(-name_label, -opts)

  column_groups <- tribble(
    ~group,       ~level1,      ~palette,
    "overall",    "Overall",    "overall",
    "expression", "Expression", "expression",
    "spatial",    "Spatial",    "spatial",
    "general",    "General (unscored)", "general"
  )

  pal <- function(name, i = 2:7) brewer.pal(9, name)[i]
  palettes <- list(
    overall     = colorRampPalette(c("#FFF3CF", "#FBC36A", "#E08A3C", "#C46A35"))(6),
    expression  = pal("Purples"),
    spatial     = colorRampPalette(c("#FCE4DC", "#C25B4E"))(6),
    transcripts = pal("Blues"),
    general     = pal("Greens"),
    general_lt  = pal("Greens", 1:4)
  )

  legends <- list(
    list(title = "Total score", palette = "overall", geom = "bar", labels = c("0", "", "0.5", "", "1"),
         size = c(0, 0.25, 0.5, 0.75, 1), color = palettes$overall[c(1, 2, 4, 5, 6)]),
    list(title = "Scaled metric", palette = "expression", geom = "funkyrect",
         labels = c("min", "", "", "", "max"), size = c(0.1, 0.3, 0.5, 0.75, 1),
         color = pal("Greys", c(2, 3, 5, 6, 7)), label_hjust = 0),
    list(palette = "spatial",     enabled = FALSE),
    list(palette = "transcripts", enabled = FALSE),
    list(palette = "general",     enabled = FALSE),
    list(palette = "general_lt",  enabled = FALSE)
  )

  g <- funky_heatmap(
    df_funky,
    column_info   = column_info,
    column_groups = column_groups,
    palettes      = palettes,
    legends       = legends,
    scale_column  = FALSE,
    position_args = position_arguments(
      expand_xmax = 4.5, expand_ymin = 0,
      expand_ymax = 0.3, col_annot_offset = 3.6, col_annot_angle = 45
    )
  )

  p <- g[[1]]
  pos <- bind_rows(lapply(p$layers, function(l) mutate(l$data[intersect(names(l$data), c("label_value", "row_id", "xmin", "xmax", "ymin", "ymax"))], across(everything(), as.character)))) %>%
    mutate(across(c(xmin, xmax, ymin, ymax), as.numeric))
  col_x <- function(i) with(filter(pos, label_value == column_info$name[column_info$id == i])[1, ], c(xmin, xmax))
  rows <- pos %>% filter(row_id %in% df_funky$id) %>% group_by(row_id) %>% summarise(y = (min(ymin) + max(ymax)) / 2)
  z <- col_x("low_vsi_pct")
  na <- bind_rows(tibble(row_id = df_funky$id[df_funky$na_ficture], x = mean(col_x("ficture_f1_score"))),
                  tibble(row_id = df_funky$id[df_funky$na_z], x = mean(z))) %>% left_join(rows, by = "row_id")
  hdr <- which(sapply(p$layers, function(l) inherits(l$geom, "GeomText")))
  p$layers[[hdr]]$data <- mutate(p$layers[[hdr]]$data, size = ifelse(label_value %in% column_groups$level1, 3.1, size))
  bar <- with(filter(pos, label_value == "Overall")[1, ], c(xmin, xmax))
  len <- df_funky$total_score * diff(bar)
  g[[1]] <- p +
    annotate("text", x = bar[1] + ifelse(len > 1.1, 0, len) + 0.08, y = rows$y[match(df_funky$id, rows$row_id)],
             label = sprintf("%.2f", df_funky$total_score), hjust = 0, size = 3.4, family = family) +
    annotate("segment", x = mean(c(z[2], col_x("pct_assigned_qced")[1])), y = min(rows$y) - 0.5, yend = max(rows$y) + 0.5,
             colour = "grey60", linewidth = 0.3) +
    annotate("text", x = na$x, y = na$y, label = "×", colour = "grey70", size = 3.5, family = family)
  g$patches$layout$heights[2] <- 0
  legs <- lapply(seq_len(length(g[[3]])), function(i) g[[3]][[i]])
  lw <- sapply(legs, `[[`, "width")
  g[[3]] <- patchwork::wrap_plots(c(legs[1], list(patchwork::plot_spacer()), legs[-1], list(patchwork::plot_spacer())),
                                  nrow = 1, widths = c(lw[1], 0.5, lw[-1], max(g[[1]]$width - sum(lw) - 0.5, 0.01)))
  g <- g & coord_equal(expand = FALSE, clip = "off") & theme(plot.margin = margin(2, 2, 2, 2))

  f <- file.path(out_dir, "benchmark_table")
  w <- g$width; h <- g$height
  g <- g & theme(plot.background = element_rect(fill = NA, colour = NA))
  g <- g + patchwork::plot_annotation(theme = theme(plot.background = element_rect(fill = NA, colour = NA)))
  f <- file.path(out_dir, "benchmark_table")
  ggsave(paste0(f, ".png"), g, width = w, height = h, dpi = 600, bg = "transparent", type = "cairo")
  ggsave(paste0(f, ".svg"), g, width = w, height = h, bg = "transparent")
}

In [30]:
main = [
    "Cellpose1 3D P VPT", "Cellpose1 3D N VPT", "Cellpose1 3D P+N VPT",
    "Cellpose1 N VPT", "Cellpose1 P VPT", "Cellpose1 P+N VPT", "Cellpose1 P Merlin",
    "Cellpose1 N Sopa", "Cellpose1 P Sopa", "Cellpose1 T Sopa", "Cellpose2 P Sopa", "Cellpose2 T Sopa",
    "CellSAM Sopa", "CellSAM P Sopa", "CellposeSAM Sopa", "CellposeSAM P Sopa",
    "Baysor", "Baysor 3D", "Spots-In-Space 3D", "Ficture Seg",
    "Proseg 3D (CP1 3D N)", "Proseg 3D (CP1 3D P)", "Proseg 3D (CP1 3D P+N)",
    "Baysor Sopa (CP1 P, 0.2)", "Ficture Seg (CP1 3D N)",
    "Raster 5μm", "Raster 10μm", "Raster 25μm", "Voronoi",
]
tables = {}
for name, dfs in dfs_by.items():
    df_all, raw = tables[name] = build_benchmark_table(dfs, exclude)
    base = main_dir / "metrics" / name / "benchmark_table"
    for df_funky, out_dir in [(df_all[df_all["id"].isin(main)], base), (df_all, base / "full")]:
        out_dir.mkdir(parents=True, exist_ok=True)
        out_dir = str(out_dir)
        print(f"{name}: {out_dir}/benchmark_table.{{png,svg}}")
        %R -i df_funky -i out_dir -i metric_labels print(plot_benchmark_heatmap(df_funky, out_dir, metric_labels))

aging: /dss/dsshome1/0C/ra98gaq/git/cellseg-benchmark/data/metrics/aging/benchmark_table/benchmark_table.{png,svg}
ℹ No row info was provided, assuming all rows in `data` are to be plotted.
ℹ Row info did not contain group information, assuming rows are ungrouped.
Scale for fill is already present.
Adding another scale for fill, which will replace the existing scale.
Coordinate system already present.
ℹ Adding new coordinate system, which will replace the existing one.
Coordinate system already present.
ℹ Adding new coordinate system, which will replace the existing one.
Coordinate system already present.
ℹ Adding new coordinate system, which will replace the existing one.
[1] "/dss/dsshome1/0C/ra98gaq/git/cellseg-benchmark/data/metrics/aging/benchmark_table/benchmark_table.svg"
aging: /dss/dsshome1/0C/ra98gaq/git/cellseg-benchmark/data/metrics/aging/benchmark_table/full/benchmark_table.{png,svg}
ℹ No row info was provided, assuming all rows in `data` are to be plotted.
ℹ Row info did 